# Search Warren Buffett's shareholder letters with ChromaDB

This notebook downloads Berkshire Hathaway's annual letters to shareholders from berkshirehathaway.com, stores them in a local Chroma vector database, and lets you search them by meaning.

Run the cells top to bottom with **Shift+Enter**. The first full run takes a few minutes (downloading about 49 letters and embedding a few thousand passages). After that, everything is cached and re-runs are quick.

The letters are copyrighted by Berkshire Hathaway; this downloads them for your own research use.

## 1. Install Chroma

In [ ]:
%pip install -q chromadb

In [ ]:
import chromadb
print("Chroma version:", chromadb.__version__)

## 2. Open the database on disk

This uses the same `chroma_db` folder as your earlier notebook, so both collections live side by side.

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
print("Existing collections:", [c.name for c in client.list_collections()])

## 3. Install the download and parsing libraries

- `requests` downloads the files
- `beautifulsoup4` reads the HTML letters (1977–1997) and the index page
- `pypdf` extracts text from the PDF letters (1998 onwards)

In [ ]:
%pip install -q requests beautifulsoup4 pypdf

## 4. Download the letters

The code reads Berkshire's letters index page to find each year's file, preferring the PDF when both exist. If the index can't be read, it tries the file-name patterns Berkshire has used over the years.

Files are saved to `buffett_letters/`, so re-running this cell skips anything already downloaded. A one-second pause between downloads keeps the load on Berkshire's server polite.

In [ ]:
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.berkshirehathaway.com/letters/"
LETTERS_DIR = Path("./buffett_letters")      # downloaded files are cached here
LETTERS_DIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (personal research notebook)"}
FIRST_YEAR, LAST_YEAR = 1977, 2025           # letters on the site start in 1977


def find_letter_links():
    """Read Berkshire's letters index page and return {year: url}, preferring PDFs."""
    links = {}
    try:
        page = requests.get(BASE_URL + "letters.html", headers=HEADERS, timeout=30)
        page.raise_for_status()
    except requests.RequestException as e:
        print("Could not read the index page, will guess file names instead:", e)
        return links
    for a in BeautifulSoup(page.text, "html.parser").find_all("a", href=True):
        href = a["href"]
        match = re.search(r"(19|20)\d{2}", href)
        if not match or not href.lower().endswith((".pdf", ".html", ".htm")):
            continue
        year = int(match.group())
        url = requests.compat.urljoin(BASE_URL, href)
        if year not in links or url.lower().endswith(".pdf"):
            links[year] = url
    return links


def candidate_urls(year):
    """File-name patterns Berkshire has used over the years, as a fallback."""
    names = [f"{year}ltr.pdf", f"{year}pdf.pdf", f"{year}.html", f"{year}htm.html"]
    return [BASE_URL + n for n in names]


def download_letter(year, url_hint=None):
    """Download one year's letter (or reuse the cached copy). Returns the file path or None."""
    cached = sorted(LETTERS_DIR.glob(f"{year}.*"))
    if cached:
        return cached[0]
    urls = ([url_hint] if url_hint else []) + candidate_urls(year)
    for url in dict.fromkeys(urls):           # de-duplicate, keep order
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
        except requests.RequestException:
            continue
        if r.status_code == 200 and len(r.content) > 2000:
            ext = ".pdf" if r.content[:4] == b"%PDF" else ".html"
            path = LETTERS_DIR / f"{year}{ext}"
            path.write_bytes(r.content)
            time.sleep(1)                     # be polite to the server
            return path
    return None


index_links = find_letter_links()
letter_files, missing = {}, []
for year in range(FIRST_YEAR, LAST_YEAR + 1):
    path = download_letter(year, index_links.get(year))
    if path:
        letter_files[year] = path
    else:
        missing.append(year)

print(f"Have {len(letter_files)} letters in {LETTERS_DIR}/")
if missing:
    print("Not found:", missing)

## 5. Extract plain text

PDF text comes out page by page; HTML text comes from the page with scripts and styling removed. Extra spaces and blank lines are tidied up.

In [ ]:
from pypdf import PdfReader


def extract_text(path):
    """Return the plain text of a PDF or HTML letter."""
    if path.suffix == ".pdf":
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    else:
        soup = BeautifulSoup(path.read_bytes(), "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        text = soup.get_text("\n")
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces
    text = re.sub(r"\n\s*\n+", "\n\n", text)     # collapse blank lines
    return text.strip()


letter_texts = {year: extract_text(path) for year, path in sorted(letter_files.items())}
for year, text in letter_texts.items():
    print(f"{year}: {len(text.split()):>6,} words")

## 6. Chunk, embed and store

Each letter is split into overlapping chunks of about 200 words (40 words of overlap, so ideas that cross a boundary aren't cut in half). Each chunk is stored with its **year** as metadata, so you can filter by year later.

Years already in the collection are skipped, so re-running this cell is safe. **This is the slow step on the first run**, since every chunk is embedded by the model on your Mac.

In [ ]:
def chunk_text(text, words_per_chunk=200, overlap=40):
    """Split text into overlapping chunks of words."""
    words = text.split()
    step = words_per_chunk - overlap
    return [" ".join(words[i:i + words_per_chunk]) for i in range(0, len(words), step)]


letters_collection = client.get_or_create_collection(
    name="buffett_letters",
    configuration={"hnsw": {"space": "cosine"}},
)

for year, text in letter_texts.items():
    already = letters_collection.get(where={"year": year}, limit=1)["ids"]
    if already:
        continue                                  # this year is already indexed
    chunks = [c for c in chunk_text(text) if len(c.split()) > 20]
    if not chunks:
        print(f"{year}: no text extracted, skipped")
        continue
    letters_collection.upsert(
        ids=[f"{year}-{i:04d}" for i in range(len(chunks))],
        documents=chunks,
        metadatas=[{"year": year, "chunk": i, "source": letter_files[year].name}
                   for i in range(len(chunks))],
    )
    print(f"{year}: stored {len(chunks)} chunks")

print("Total chunks in collection:", letters_collection.count())

## 7. A search helper

`ask_buffett()` searches by meaning and prints the year, a similarity score (higher is closer), and the passage.

Options:
- `n_results`: how many passages to return
- `year_from` / `year_to`: limit to a range of years
- `contains`: require an exact word or name in the passage, such as `"GEICO"`
- `full=True`: print whole passages instead of the first 600 characters

In [ ]:
import textwrap


def ask_buffett(question, n_results=5, year_from=None, year_to=None, contains=None, full=False):
    """Search the letters by meaning, optionally limited to a year range or an exact word."""
    conditions = []
    if year_from is not None:
        conditions.append({"year": {"$gte": year_from}})
    if year_to is not None:
        conditions.append({"year": {"$lte": year_to}})
    where = conditions[0] if len(conditions) == 1 else ({"$and": conditions} if conditions else None)

    results = letters_collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where,
        where_document={"$contains": contains} if contains else None,
    )
    print(f"Q: {question}\n")
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        text = doc if full else doc[:600] + ("..." if len(doc) > 600 else "")
        print(f"[{meta['year']}]  similarity {1 - dist:.2f}")
        print(textwrap.fill(text, width=110))
        print("-" * 110)

## 8. Ask questions in plain English

In [ ]:
ask_buffett("What does Buffett think about share buybacks?")
ask_buffett("Why is insurance float valuable to Berkshire?")
ask_buffett("What mistakes did he admit to?", n_results=3)
ask_buffett("How does he judge the quality of a company's management?", n_results=3)

## 9. Narrow the search

Year ranges are useful for seeing how his thinking changed over time. The `contains` filter is useful for names and exact terms that meaning-based search can match loosely.

In [ ]:
ask_buffett("Thoughts on derivatives and their risks", year_from=2000, year_to=2010, n_results=3)
ask_buffett("Why did Berkshire buy this business?", contains="GEICO", n_results=3)

## 10. Which years discuss a topic most

This takes the 40 most relevant passages for a topic and counts them by year: a quick map of when a subject came up.

In [ ]:
from collections import Counter


def topic_by_year(topic, n_results=40):
    """Count which years the most relevant passages come from."""
    results = letters_collection.query(query_texts=[topic], n_results=n_results)
    counts = Counter(meta["year"] for meta in results["metadatas"][0])
    print(f"Top {n_results} passages about '{topic}', by year:")
    for year, count in sorted(counts.items()):
        print(f"  {year}: {'#' * count} ({count})")


topic_by_year("succession planning after Buffett")

## Tips

- **Ask the way you'd ask a person.** "How does Buffett decide what price to pay for a business?" works better than single keywords.
- **Compare eras:** run the same question with `year_from=1977, year_to=1995` and then `year_from=2010`.
- **Read the source:** each result shows its year, so you can open the full letter in `buffett_letters/` for context.
- **Next step, RAG:** pass the top passages and your question to an AI model, and ask it to answer using only those passages, citing years.